# Itération 4 — clauses sous forme de segments, sans les catégories d'identité

**Projet Big Data & Deep Learning — Agent d'audit de conformité documentaire**

## Ce que l'itération 3 a appris

Sur 8 contrats réels, l'itération 3 (classe « Autre ») éliminait les fantômes (6,5 %) mais ne retrouvait que 33 % des clauses.
Le diagnostic montre deux causes :

1. **Les catégories d'identité** (nom du contrat, parties, date de signature, date d'entrée en vigueur) sont de petits passages
   dans l'en-tête ou le bloc de signature : retrouvées 5 fois sur 35. Elles sont maintenant traitées par un **extracteur à règles**
   (`src/data/header.py`, mesuré sur 102 contrats) : le classifieur ne s'en occupe plus, et les blocs de signature deviennent « Autre »
   sans contradiction avec les blocs annotés « Parties ».
2. **Forme des exemples** : le modèle apprenait sur des passages de 194 caractères mais reçoit des segments de 471. On ajoute donc,
   pour chaque clause annotée, le **segment qui la contient**.

## Ce qui change

* 37 catégories de clauses (les 41 moins les 4 d'identité) + « Autre » = **38 classes**
* entraînement : passages isolés + clauses en segments + 6 000 paragraphes « Autre » ; test : mêmes types, **contrats différents**
* tout le reste est identique : même modèle, poids de classes, arrêt automatique, graine

## Mode d'emploi

1. **Exécution** > **Modifier le type d'exécution** > **T4 GPU** > Enregistrer
2. **Exécution** > **Tout exécuter**
3. À l'envoi des fichiers, dépose les **SIX** fichiers de `C:\projets\audit-conformite-mlops\data\processed` :
   `clauses_train`, `clauses_test`, `segments_train`, `segments_test`, `autre_train`, `autre_test` (`.jsonl`)

> Si Colab **redémarre tout seul** après la cellule d'installation, c'est normal : relance à partir de la cellule suivante.

Durée : environ **25 à 35 minutes**.

## 1. GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU détecté :", torch.cuda.get_device_name(0))
else:
    print("AUCUN GPU. Exécution > Modifier le type d'exécution > T4 GPU, puis relance.")

## 2. Installation

In [ ]:
import sys
print("Python", sys.version.split()[0])
!pip install "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub<1.0" "accelerate>=1.1,<2" scikit-learn 2>&1 | tail -6

In [ ]:
# Vérifie que la bonne version est INSTALLÉE et CHARGÉE (Colab fournit transformers 5, qui est incompatible
# avec ce notebook : une version déjà chargée en mémoire masquerait l'installation).
import sys, os, importlib.metadata as md

on_disk = md.version("transformers")
loaded = sys.modules["transformers"].__version__ if "transformers" in sys.modules else None
print(f"transformers installé : {on_disk}  |  chargé en mémoire : {loaded}")
if on_disk != "4.46.3":
    raise SystemExit("La version 4.46.3 n'a pas pu être installée : copie l'affichage de la cellule précédente.")
if loaded not in (None, "4.46.3"):
    print("\nLa session utilisait une autre version : redémarrage automatique (normal).")
    print("-> Quand Colab a redémarré, relance à partir de la CELLULE 3 (pas besoin de refaire celle-ci).")
    os.kill(os.getpid(), 9)

# Test d'import immédiat : s'il y a un problème, on le voit ici en 5 secondes, pas après 25 minutes.
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
print("Imports OK : transformers", __import__("transformers").__version__)

## 3. Dépôt des données

Sélectionne les **SIX** fichiers d'un coup.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
NEEDED = ("clauses_train.jsonl", "clauses_test.jsonl", "segments_train.jsonl", "segments_test.jsonl",
          "autre_train.jsonl", "autre_test.jsonl")
for name in NEEDED:
    if os.path.exists(name):
        n = sum(1 for _ in open(name, encoding="utf-8"))
        print(f"OK   {name:22s} {n:>7,} lignes")
    else:
        print(f"MANQUANT : {name}  -> relance cette cellule et choisis les bons fichiers")

## 4. Données : train, validation, test

Le classifieur ne traite plus les 4 catégories d'identité. Trois types de lignes, **le test venant de contrats différents** : passages isolés (`span`), clauses prises dans leur segment (`segment`), paragraphes neutres (`autre`).

In [ ]:
import os
import torch
import json
import numpy as np
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

NEUTRAL = "Autre"
HEADER = {"Document Name", "Parties", "Agreement Date", "Effective Date"}     # extracteur à règles, pas le classifieur

def load_split(path, kind):
    X, y, k = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            lab = row.get("label")
            if lab and lab not in HEADER:
                X.append(row["text"]); y.append(lab); k.append(kind)
    return X, y, k

def combine(*parts):
    return [sum((p[i] for p in parts), []) for i in range(3)]

X_full, y_full_raw, kind_full = combine(load_split("clauses_train.jsonl", "span"),
                                        load_split("segments_train.jsonl", "segment"),
                                        load_split("autre_train.jsonl", "autre"))
X_test, y_test_raw, kind_test = combine(load_split("clauses_test.jsonl", "span"),
                                        load_split("segments_test.jsonl", "segment"),
                                        load_split("autre_test.jsonl", "autre"))

encoder = LabelEncoder().fit(y_full_raw)
known = set(encoder.classes_)
keep = [i for i, l in enumerate(y_test_raw) if l in known]
X_test = [X_test[i] for i in keep]
kind_test = np.array([kind_test[i] for i in keep])
y_test = encoder.transform([y_test_raw[i] for i in keep]).tolist()
y_full = encoder.transform(y_full_raw).tolist()
n_labels = len(encoder.classes_)
AUTRE_ID = int(encoder.transform([NEUTRAL])[0])

counts = Counter(y_full)
strat = y_full if min(counts.values()) >= 2 else None
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.10, random_state=42, stratify=strat)

# Catégories rares : moins de 100 PASSAGES ISOLÉS d'entraînement (même définition qu'aux itérations précédentes).
span_counts = Counter(y for y, k in zip(y_full, kind_full) if k == "span")
RARE = sorted(c for c in range(n_labels) if c != AUTRE_ID and span_counts.get(c, 0) < 100)

def n_kind(kinds, k): return int(sum(1 for x in kinds if x == k))
print(f"train      : {len(X_train):,}  (passages {n_kind(kind_full, 'span'):,} | segments {n_kind(kind_full, 'segment'):,} | Autre {n_kind(kind_full, 'autre'):,}, avant découpage validation)")
print(f"validation : {len(X_val):,}")
print(f"test       : {len(X_test):,}  (passages {n_kind(kind_test, 'span'):,} | segments {n_kind(kind_test, 'segment'):,} | Autre {n_kind(kind_test, 'autre'):,})")
print(f"classes    : {n_labels}  (37 catégories + « Autre », dont {len(RARE)} rares)")
assert n_labels == 38, "On attend 38 classes : 37 catégories de clauses + Autre."

## 5. Poids des classes

Même formule qu'à l'itération 2 : racine de (effectif moyen / effectif de la classe). « Autre », très fréquente, reçoit un poids faible.

In [ ]:
import numpy as np
import torch

train_counts = np.bincount(y_train, minlength=n_labels).astype(float)
train_counts[train_counts == 0] = 1.0
weights = np.sqrt(train_counts.mean() / train_counts)
weights = weights / weights.mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float)

print(f"Poids de « Autre » : {weights[AUTRE_ID]:.2f}")
order = np.argsort(weights)
print("Poids les plus forts (catégories rares) :")
for i in order[-3:]:
    print(f"   {encoder.classes_[i][:40]:40s} {weights[i]:.2f}")

## 6. Configuration

In [ ]:
MODELS = [
    {"key": "legalbert-v4", "name": "nlpaueb/legal-bert-base-uncased",
     "label": "Legal-BERT v4 (segments, sans les catégories d'identité)", "fp16": True, "lr": 3e-5},
]
MAX_EPOCHS = 10
PATIENCE   = 2
BATCH_SIZE = 32
MAX_LENGTH = 256
SEED       = 42

# Références (mesurées sur les passages isolés de test) : baseline TF-IDF + SVM et itérations précédentes
BASELINE_F1, V2_F1_MACRO, V3_F1_MACRO = 0.670, 0.719, 0.735
RESULTS_FILE = "benchmark_v4.json"
print(f"{len(MODELS)} modèle | jusqu'à {MAX_EPOCHS} époques | arrêt après {PATIENCE} époques sans progrès")

## 7. Entraînement pondéré avec arrêt automatique

In [ ]:
import json
import os
import numpy as np
import torch
import gc, time, shutil
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score


class ClauseDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


class WeightedTrainer(Trainer):
    """Trainer dont la perte pénalise davantage les erreurs sur les classes rares."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(outputs.logits.device))
        loss = loss_fn(outputs.logits.float(), labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"f1_macro": f1_score(p.label_ids, preds, average="macro", zero_division=0)}


def run_model(cfg):
    set_seed(SEED)
    print(f"\n{'=' * 64}\n  {cfg['label']}  ({cfg['name']})\n{'=' * 64}")
    tok = AutoTokenizer.from_pretrained(cfg["name"])
    enc = lambda X: tok(X, truncation=True, padding="max_length", max_length=MAX_LENGTH)
    ds_tr, ds_va, ds_te = (ClauseDataset(enc(X), y) for X, y in ((X_train, y_train), (X_val, y_val), (X_test, y_test)))

    model = AutoModelForSequenceClassification.from_pretrained(cfg["name"], num_labels=n_labels)
    out_dir = f"ckpt_{cfg['key']}"
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=cfg["lr"], warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        logging_steps=50, fp16=cfg["fp16"], report_to=[], seed=SEED,
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0

    evals = [h for h in trainer.state.log_history if "eval_f1_macro" in h]
    best_epoch = int(max(evals, key=lambda h: h["eval_f1_macro"])["epoch"])
    epochs_run = int(evals[-1]["epoch"])

    # Le modèle est sauvegardé AVANT les mesures : une erreur de mesure ne fait pas perdre l'entraînement.
    trainer.save_model(f"model_{cfg['key']}")
    tok.save_pretrained(f"model_{cfg['key']}")

    # Évaluation finale sur le TEST, jamais vu pendant l'entraînement.
    t0 = time.time()
    preds = np.argmax(trainer.predict(ds_te).predictions, axis=-1)
    infer_s = time.time() - t0
    y_arr = np.array(y_test)
    is_span, is_seg, is_autre = kind_test == "span", kind_test == "segment", y_arr == AUTRE_ID

    def macro(mask):
        yt, yp = y_arr[mask], preds[mask]
        return (f1_score(yt, yp, labels=sorted(set(yt.tolist())), average="macro", zero_division=0),
                accuracy_score(yt, yp), float((yp == AUTRE_ID).mean()))

    f1_span, acc_span, lost_span = macro(is_span)        # passages isolés : comparable aux itérations 2 et 3
    f1_seg, acc_seg, lost_seg = macro(is_seg)            # clauses dans leur segment : ce que le pipeline voit
    rare_mask = is_span & np.isin(y_arr, RARE)
    f1_rare = f1_score(y_arr[rare_mask], preds[rare_mask], labels=RARE, average="macro", zero_division=0)

    tp = int((is_autre & (preds == AUTRE_ID)).sum())
    autre_recall = tp / max(int(is_autre.sum()), 1)
    autre_precision = tp / max(int((preds == AUTRE_ID).sum()), 1)

    res = {
        "key": cfg["key"], "label": cfg["label"], "model_name": cfg["name"], "iteration": 4,
        "f1_macro": float(f1_span), "f1_macro_rare": float(f1_rare), "accuracy": float(acc_span),
        "f1_macro_segments": float(f1_seg), "accuracy_segments": float(acc_seg),
        "clauses_perdues_passages": lost_span, "clauses_perdues_segments": lost_seg,
        "autre_rappel": float(autre_recall), "autre_precision": float(autre_precision),
        "fantomes_classification": float(1 - autre_recall),
        "train_minutes": round(train_s / 60, 2),
        "latency_ms_per_clause": round(1000 * infer_s / len(X_test), 2),
        "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        "best_epoch": best_epoch, "epochs_run": epochs_run, "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE, "learning_rate": cfg["lr"], "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH, "fp16": cfg["fp16"], "seed": SEED,
        "class_weighting": "sqrt_inverse_frequency",
        "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test),
        "n_classes": n_labels,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    np.save(f"preds_{cfg['key']}.npy", preds)
    shutil.rmtree(out_dir, ignore_errors=True)

    print(f"\n  passages isolés : F1 macro {f1_span:.3f} | rares {f1_rare:.3f} | accuracy {acc_span:.3f} | perdus dans « Autre » {100 * lost_span:.1f} %")
    print(f"  clauses en segment : F1 macro {f1_seg:.3f} | accuracy {acc_seg:.3f} | perdues dans « Autre » {100 * lost_seg:.1f} %")
    print(f"  « Autre »          : rappel {autre_recall:.3f} | précision {autre_precision:.3f}")
    print(f"  meilleure époque {best_epoch}/{epochs_run}")
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return res


results = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else []
done = {r["key"] for r in results}
for cfg in MODELS:
    if cfg["key"] in done:
        continue
    try:
        results.append(run_model(cfg))
        json.dump(results, open(RESULTS_FILE, "w"), indent=2)
    except Exception as e:
        print(f"\nÉCHEC sur {cfg['label']} : {type(e).__name__}: {e}")
        gc.collect(); torch.cuda.empty_cache()

print(f"\n{len(results)} / {len(MODELS)} modèle terminé.")

## 8. Verdict : itération 2 contre itération 3

In [ ]:
import json
best = max(results, key=lambda r: r["f1_macro_segments"])
print("=" * 72)
print(f"  Baseline TF-IDF + SVM (41 catégories, passages) : F1 macro {BASELINE_F1:.3f}")
print(f"  Itération 2 / 3 (41 catégories, passages)       : {V2_F1_MACRO:.3f} / {V3_F1_MACRO:.3f}")
print(f"  Itération 4 (37 catégories), passages isolés    : F1 macro {best['f1_macro']:.3f}   rares {best['f1_macro_rare']:.3f}")
print(f"  Itération 4, clauses DANS LEUR SEGMENT          : F1 macro {best['f1_macro_segments']:.3f}   "
      f"(perdues dans « Autre » : {100 * best['clauses_perdues_segments']:.1f} %)")
print("-" * 72)
print(f"  « Autre » : rappel {best['autre_rappel']:.3f} | précision {best['autre_precision']:.3f}")
print("=" * 72)
print("Les 41 catégories ne sont pas comparables aux 37 : l'extracteur traite les 4 autres.")
print("Le vrai verdict vient de l'essai de bout en bout sur de vrais contrats :")
print("   python scripts/eval_pipeline.py --model models/best_model_v4 --contracts 8")

## 9. Téléchargement

Le modèle est zippé avec ses étiquettes (42, dont « Autre ») et ses métriques.

In [ ]:
import json
import shutil
from google.colab import files
shutil.rmtree("best_model_v4", ignore_errors=True)
shutil.copytree(f"model_{best['key']}", "best_model_v4")
json.dump(list(encoder.classes_), open("best_model_v4/labels.json", "w"), indent=2, ensure_ascii=False)
json.dump(best, open("best_model_v4/metrics.json", "w"), indent=2)
!zip -qr best_model_v4.zip best_model_v4

for f in ("benchmark_v4.json", "best_model_v4.zip"):
    files.download(f)